# ICT-4442 Deep Learning Mini Project: Interim Submission
## Stock Market Trend Prediction using BiLSTM and XGBoost

**Scope of this notebook (interim):** data ingestion, technical-indicator features, preprocessing and scaling, 3D sliding-window tensors for the BiLSTM, and the XGBoost baseline.

**Target:** next-day close price (and the implied direction of movement).

**Evaluation metrics**

$$\text{RMSE}=\sqrt{\frac{1}{n}\sum_{i=1}^{n}\left(y_i-\hat{y}_i\right)^2}$$

$$\text{MAE}=\frac{1}{n}\sum_{i=1}^{n}\left|y_i-\hat{y}_i\right|$$

$$\text{DA}=\frac{1}{n}\sum_{i=1}^{n}\mathbb{1}\left[\operatorname{sign}(\hat{y}_i-C_i)=\operatorname{sign}(y_i-C_i)\right]\times 100\%$$

where $C_i$ is the close on day $i$, $y_i$ is the actual close on day $i+1$, and $\hat{y}_i$ is the predicted close on day $i+1$.

## 0. Setup
Installs dependencies and sets the global configuration. All tunable values (ticker, start date, window length, split ratio) live in one place.

In [ ]:
!pip -q install yfinance xgboost

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from numpy.lib.stride_tricks import sliding_window_view
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from xgboost import XGBRegressor

# ----- Global configuration -----
TICKER     = "^NSEI"        # NIFTY 50  (use "^BSESN" for SENSEX)
START_DATE = "2013-01-01"   # gives 12+ years of daily data
END_DATE   = None           # None = up to today
WINDOW     = 60             # sequence length (days) for the BiLSTM tensors
TEST_FRAC  = 0.20           # last 20% of the timeline is held out for testing
SEED       = 42

np.random.seed(SEED)

## 1. Data Ingestion Pipeline
**What it does:** downloads daily OHLCV data for the NIFTY 50 from Yahoo Finance using `yfinance`, flattens the column index returned by newer `yfinance` versions, sorts by date, removes duplicate dates, and checks that the history covers at least 10 years.

In [ ]:
def fetch_data(ticker: str, start: str, end: str = None) -> pd.DataFrame:
    """Download daily OHLCV data and return a clean, date-indexed DataFrame."""
    raw = yf.download(ticker, start=start, end=end,
                      interval="1d", auto_adjust=False, progress=False)
    if raw.empty:
        raise ValueError(f"No data returned for {ticker}. Check the ticker or your connection.")

    # Newer yfinance versions return MultiIndex columns like ('Close', '^NSEI')
    if isinstance(raw.columns, pd.MultiIndex):
        raw.columns = raw.columns.get_level_values(0)

    df = raw[["Open", "High", "Low", "Close", "Volume"]].copy()
    df.index = pd.to_datetime(df.index)
    df = df[~df.index.duplicated(keep="first")].sort_index()
    return df


df_raw = fetch_data(TICKER, START_DATE, END_DATE)

years = (df_raw.index[-1] - df_raw.index[0]).days / 365.25
assert years >= 10, f"Only {years:.1f} years of data; need 10+."

print(f"Ticker : {TICKER}")
print(f"Range  : {df_raw.index[0].date()} to {df_raw.index[-1].date()} ({years:.1f} years)")
print(f"Rows   : {len(df_raw)}")
df_raw.tail()

## 2. Technical Indicator Feature Engineering
**What it does:** computes four families of indicators directly in pandas (no extra dependency) and appends them as columns.

| Indicator | Formula | Periods |
|---|---|---|
| SMA | $\text{SMA}_n=\frac{1}{n}\sum_{i=0}^{n-1}C_{t-i}$ | 20, 50 |
| EMA | $\text{EMA}_t=\alpha C_t+(1-\alpha)\,\text{EMA}_{t-1},\quad \alpha=\frac{2}{n+1}$ | 12, 26 |
| RSI | $\text{RSI}=100-\frac{100}{1+RS},\quad RS=\frac{\text{avg gain}}{\text{avg loss}}$ (Wilder smoothing) | 14 |
| MACD | $\text{MACD}=\text{EMA}_{12}-\text{EMA}_{26}$, signal $=\text{EMA}_9(\text{MACD})$, histogram $=\text{MACD}-\text{signal}$ | 12, 26, 9 |

A one-day return is added as well, since returns are closer to stationary than raw prices.

In [ ]:
def add_indicators(df: pd.DataFrame) -> pd.DataFrame:
    """Append RSI, MACD, EMA, SMA and daily return to an OHLCV DataFrame."""
    out = df.copy()
    close = out["Close"]

    # --- Simple Moving Averages ---
    out["SMA_20"] = close.rolling(window=20).mean()
    out["SMA_50"] = close.rolling(window=50).mean()

    # --- Exponential Moving Averages ---
    out["EMA_12"] = close.ewm(span=12, adjust=False).mean()
    out["EMA_26"] = close.ewm(span=26, adjust=False).mean()

    # --- RSI (14) with Wilder smoothing, alpha = 1/14 ---
    delta     = close.diff()
    gain      = delta.clip(lower=0)
    loss      = -delta.clip(upper=0)
    avg_gain  = gain.ewm(alpha=1/14, adjust=False, min_periods=14).mean()
    avg_loss  = loss.ewm(alpha=1/14, adjust=False, min_periods=14).mean()
    rs        = avg_gain / avg_loss.replace(0, np.nan)
    out["RSI_14"] = 100 - (100 / (1 + rs))

    # --- MACD (12, 26, 9) ---
    out["MACD"]        = out["EMA_12"] - out["EMA_26"]
    out["MACD_signal"] = out["MACD"].ewm(span=9, adjust=False).mean()
    out["MACD_hist"]   = out["MACD"] - out["MACD_signal"]

    # --- Daily return ---
    out["Return_1d"] = close.pct_change()
    return out


df_feat = add_indicators(df_raw)
df_feat.tail()

## 3. Data Preprocessing & Scaling
**What it does:**
1. Treats zero volume (a known data gap in early index history) as missing and forward-fills it.
2. Replaces infinities, then drops the leading NaN rows created by indicator warm-up periods.
3. Builds the target: next-day close $C_{t+1}$ and next-day return $r_{t+1}=\frac{C_{t+1}}{C_t}-1$.
4. Splits **chronologically** (no shuffling) into train and test.
5. Fits `MinMaxScaler` on the **training rows only**, then applies it to all rows:

$$x'=\frac{x-x_{\min}^{\text{train}}}{x_{\max}^{\text{train}}-x_{\min}^{\text{train}}}$$

Fitting on train only prevents data leakage. Test values can fall slightly outside $[0,1]$ when the market makes new highs, which is expected and correct.

In [ ]:
FEATURE_COLS = ["Open", "High", "Low", "Close", "Volume",
                "SMA_20", "SMA_50", "EMA_12", "EMA_26",
                "RSI_14", "MACD", "MACD_signal", "MACD_hist", "Return_1d"]

df = df_feat.copy()

# --- Missing value handling ---
df["Volume"] = df["Volume"].replace(0, np.nan).ffill()
df = df.replace([np.inf, -np.inf], np.nan)
print("Missing values before cleaning:\n", df[FEATURE_COLS].isna().sum()[lambda s: s > 0], "\n")

# --- Targets (shifted so row t holds the value for day t+1) ---
df["Target_Close"]  = df["Close"].shift(-1)
df["Target_Return"] = df["Target_Close"] / df["Close"] - 1
df["Target_Dir"]    = (df["Target_Close"] > df["Close"]).astype(int)   # 1 = up, 0 = down/flat

df = df.dropna().copy()
print(f"Rows after dropna: {len(df)}  |  NaNs remaining: {int(df.isna().sum().sum())}")

# --- Chronological split on rows ---
split_idx = int(len(df) * (1 - TEST_FRAC))
print(f"Train rows: {split_idx}  ({df.index[0].date()} to {df.index[split_idx-1].date()})")
print(f"Test rows : {len(df) - split_idx}  ({df.index[split_idx].date()} to {df.index[-1].date()})")

# --- Scale: fit on TRAIN only, transform everything ---
scaler = MinMaxScaler(feature_range=(0, 1))
scaler.fit(df[FEATURE_COLS].iloc[:split_idx])
X_scaled = scaler.transform(df[FEATURE_COLS])          # shape: (rows, features)
print("Scaled feature matrix:", X_scaled.shape)

## 4. 3D Sliding-Window Tensor Formatting
**What it does:** converts the 2D table `(rows, features)` into a 3D tensor `(samples, time_steps, features)` for the BiLSTM. Each sample is the previous `WINDOW` (60) trading days, and its label is the **next-day** value for the last day in that window.

$$X_k=\left[x_{k},\,x_{k+1},\,\dots,\,x_{k+59}\right]\;\longrightarrow\;y_k=\text{target}_{k+59}$$

The function uses NumPy's `sliding_window_view`, so no Python loop is needed. The train/test split on windows is aligned to the same date boundary used above, so no test-period target ever appears in training.

In [ ]:
def create_sequences(features: np.ndarray, targets: np.ndarray, window: int):
    """
    Turn 2D features (rows, n_features) into 3D windows (samples, window, n_features).
    The label of each window is the target stored at the window's LAST row.
    """
    assert len(features) == len(targets), "features and targets must align"
    win = sliding_window_view(features, window_shape=window, axis=0)  # (n-w+1, n_features, window)
    X_seq = np.ascontiguousarray(win.transpose(0, 2, 1))              # (samples, window, n_features)
    y_seq = targets[window - 1:]                                      # (samples,)
    return X_seq, y_seq


# Labels for the upcoming BiLSTM: next-day return (regression) and direction (classification)
X_seq, y_ret_seq   = create_sequences(X_scaled, df["Target_Return"].values, WINDOW)
_,     y_dir_seq   = create_sequences(X_scaled, df["Target_Dir"].values,    WINDOW)
_,     y_close_seq = create_sequences(X_scaled, df["Target_Close"].values,  WINDOW)
seq_dates          = df.index[WINDOW - 1:]          # date of the last day inside each window

# Windows whose last day is before split_idx belong to train; the rest to test
n_train_seq = split_idx - (WINDOW - 1)
X_train_seq, X_test_seq = X_seq[:n_train_seq], X_seq[n_train_seq:]
y_ret_train, y_ret_test = y_ret_seq[:n_train_seq], y_ret_seq[n_train_seq:]
y_dir_train, y_dir_test = y_dir_seq[:n_train_seq], y_dir_seq[n_train_seq:]

print("3D tensor shape (samples, time_steps, features):", X_seq.shape)
print("X_train_seq:", X_train_seq.shape, "| X_test_seq:", X_test_seq.shape)
print("y_ret_train:", y_ret_train.shape, "| y_ret_test:", y_ret_test.shape)
print(f"Class balance (up days) train: {y_dir_train.mean():.3f} | test: {y_dir_test.mean():.3f}")

# Sanity check: first window equals the first 60 scaled rows
assert np.allclose(X_seq[0], X_scaled[:WINDOW])

## 5. Baseline Model: XGBoost Regressor
**What it does:** trains an `XGBRegressor` on the engineered (scaled) tabular features, using the same chronological split.

**Design choice:** tree models cannot predict values outside the range seen in training, and an index that trends upward keeps setting new highs. So the model predicts the **next-day return** and the price is rebuilt as

$$\hat{C}_{t+1}=C_t\,(1+\hat{r}_{t+1})$$

RMSE and MAE are reported on this reconstructed **price** (in index points). Directional accuracy compares the predicted and actual direction of the move. The last 10% of the training period is used as a time-ordered validation set for early stopping. A **persistence baseline** ($\hat{C}_{t+1}=C_t$) is included as a sanity reference, since beating it on next-day price is difficult.

In [ ]:
# --- Tabular split (same boundary as the sequences) ---
X_train, X_test = X_scaled[:split_idx], X_scaled[split_idx:]
y_train = df["Target_Return"].values[:split_idx]
y_test  = df["Target_Return"].values[split_idx:]

close_test        = df["Close"].values[split_idx:]          # C_t on each test day
actual_next_close = df["Target_Close"].values[split_idx:]   # C_{t+1}
test_dates        = df.index[split_idx:]

# --- Time-ordered validation slice for early stopping ---
val_cut = int(len(X_train) * 0.9)
X_tr, X_val = X_train[:val_cut], X_train[val_cut:]
y_tr, y_val = y_train[:val_cut], y_train[val_cut:]

xgb = XGBRegressor(
    n_estimators=1000,
    learning_rate=0.03,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    objective="reg:squarederror",
    early_stopping_rounds=50,
    random_state=SEED,
    n_jobs=-1,
)
xgb.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
print("Best iteration:", xgb.best_iteration)

# --- Predict return, rebuild price ---
pred_ret   = xgb.predict(X_test)
pred_close = close_test * (1 + pred_ret)


def directional_accuracy(close_t, actual_next, pred_next):
    """Percentage of days where predicted and actual move direction agree."""
    return np.mean(np.sign(pred_next - close_t) == np.sign(actual_next - close_t)) * 100


def evaluate(name, close_t, actual_next, pred_next, with_da=True):
    rmse = np.sqrt(mean_squared_error(actual_next, pred_next))
    mae  = mean_absolute_error(actual_next, pred_next)
    da   = directional_accuracy(close_t, actual_next, pred_next) if with_da else np.nan
    return {"Model": name, "RMSE": rmse, "MAE": mae, "Directional Acc. (%)": da}


results = pd.DataFrame([
    evaluate("XGBoost (return -> price)", close_test, actual_next_close, pred_close),
    evaluate("Persistence (C_t+1 = C_t)",  close_test, actual_next_close, close_test, with_da=False),
]).set_index("Model").round(3)

print("\nTest-set performance (index points):")
results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))

# Predicted vs actual (zoomed on the last 150 test days for readability)
k = 150
axes[0].plot(test_dates[-k:], actual_next_close[-k:], label="Actual close", linewidth=1.6)
axes[0].plot(test_dates[-k:], pred_close[-k:], label="XGBoost predicted", linewidth=1.2)
axes[0].set_title(f"{TICKER}: next-day close, last {k} test days")
axes[0].legend(); axes[0].grid(alpha=0.3)

# Feature importance
imp = pd.Series(xgb.feature_importances_, index=FEATURE_COLS).sort_values()
imp.plot.barh(ax=axes[1])
axes[1].set_title("XGBoost feature importance")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Interim Summary and Next Steps

**Completed:** a reproducible pipeline that downloads 10+ years of NIFTY 50 daily data, engineers SMA, EMA, RSI and MACD features, cleans and scales them without leakage, produces `(samples, 60, 14)` tensors ready for the BiLSTM, and trains an XGBoost baseline evaluated with RMSE, MAE and directional accuracy.

**Key methodological points**
- Chronological split only. Shuffling would leak future information into training.
- `MinMaxScaler` is fitted on the training period only.
- XGBoost predicts next-day return and the price is reconstructed, because trees cannot extrapolate beyond the training price range.
- Next-day price is close to a random walk, so the persistence baseline is hard to beat on RMSE/MAE. Directional accuracy slightly above 50% is a realistic baseline outcome.

**Remaining for the final submission**
1. Build and train the BiLSTM on `X_train_seq` with `y_ret_train` (regression) or `y_dir_train` (direction).
2. Compare BiLSTM against XGBoost and persistence on the same test period.
3. Hyperparameter tuning, walk-forward validation, and a discussion of limitations.